# Data Quality Check – Chicago Rideshare Trips

Goal: find problems in the trip data before we use it, and decide how to clean it.

Data used: the full City of Chicago Transportation Network Providers dataset: 628,189,680 trips from November 2018 to 2026, stored as Parquet. An earlier version of this check used a 200,000-trip sample; where the full data gives a different answer, we say so below.

In [1]:
from pathlib import Path
from pyspark.sql import SparkSession, functions as F

# find the project root (the folder that has data/ in it), wherever Jupyter started
ROOT = Path.cwd()
while not (ROOT / "data").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
print("project root:", ROOT)

spark = (SparkSession.builder.appName("data-quality-full")
         .config("spark.driver.memory", "8g")
         .config("spark.ui.showConsoleProgress", "false")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
df = spark.read.option("mergeSchema", "true").parquet(str(ROOT / "data" / "parquet" / "tnp"))
df.printSchema()
print(df.count())

num_cols = ["trip_seconds", "trip_miles", "percent_time_chicago", "percent_distance_chicago",
            "fare", "tip", "additional_charges", "trip_total", "trips_pooled",
            "pickup_centroid_latitude", "pickup_centroid_longitude",
            "dropoff_centroid_latitude", "dropoff_centroid_longitude"]

for c in num_cols:
    df = df.withColumn(c, F.col(c).cast("double"))

project root: /Users/dharmikpatel/Documents/ride-or-rail


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/01 18:23:22 WARN Utils: Your hostname, Dharmiks-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 10.0.0.246 instead (on interface en0)
26/11/01 18:23:22 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/Users/dharmikpatel/Documents/ride-or-rail/.venv/lib/python3.13/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/11/01 18:23:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


root
 |-- trip_id: string (nullable = true)
 |-- trip_start_timestamp: timestamp (nullable = true)
 |-- trip_end_timestamp: timestamp (nullable = true)
 |-- trip_seconds: integer (nullable = true)
 |-- trip_miles: double (nullable = true)
 |-- pickup_census_tract: string (nullable = true)
 |-- dropoff_census_tract: string (nullable = true)
 |-- pickup_community_area: integer (nullable = true)
 |-- dropoff_community_area: integer (nullable = true)
 |-- fare: double (nullable = true)
 |-- tip: double (nullable = true)
 |-- additional_charges: double (nullable = true)
 |-- trip_total: double (nullable = true)
 |-- shared_trip_authorized: string (nullable = true)
 |-- trips_pooled: integer (nullable = true)
 |-- pickup_centroid_latitude: double (nullable = true)
 |-- pickup_centroid_longitude: double (nullable = true)
 |-- pickup_centroid_location: string (nullable = true)
 |-- dropoff_centroid_latitude: double (nullable = true)
 |-- dropoff_centroid_longitude: double (nullable = true)
 |-

The data loaded correctly: 200,000 rows and 26 columns.

Problem found: almost all number columns (trip_seconds, trip_miles, fare, tip, etc.) are stored as text, not numbers. We need to convert them before doing any math.

In [2]:
df.select(num_cols).describe().show(vertical=True)

-RECORD 0-----------------------------------------
 summary                    | count               
 trip_seconds               | 627960198           
 trip_miles                 | 628182553           
 percent_time_chicago       | 327836628           
 percent_distance_chicago   | 327824967           
 fare                       | 627129262           
 tip                        | 627129370           
 additional_charges         | 627129261           
 trip_total                 | 627129250           
 trips_pooled               | 628189680           
 pickup_centroid_latitude   | 577060309           
 pickup_centroid_longitude  | 577060309           
 dropoff_centroid_latitude  | 573535627           
 dropoff_centroid_longitude | 573535627           
-RECORD 1-----------------------------------------
 summary                    | mean                
 trip_seconds               | 1117.3337566595264  
 trip_miles                 | 6.8772252246870975  
 percent_time_chicago       | 0

## Numbers summary

After converting to numbers, a few things look wrong:
- Some trips have 0 seconds, 0 miles, or a $0 fare.
- percent_time_chicago and percent_distance_chicago go above 1.0, even though they should be between 0 and 1.
- Some extreme values: longest trip about 9.6 hours, longest about 270 miles, highest total $518. The average trip is about 21 minutes and 8 miles.

In [3]:
total = df.count()
nulls = df.select([
    F.round(F.count(F.when(F.col(c).isNull(), c)) / total * 100, 2).alias(c)
    for c in df.columns
])
nulls.show(vertical=True)

-RECORD 0---------------------------
 trip_id                    | 0.0   
 trip_start_timestamp       | 0.0   
 trip_end_timestamp         | 0.0   
 trip_seconds               | 0.04  
 trip_miles                 | 0.0   
 pickup_census_tract        | 38.54 
 dropoff_census_tract       | 38.81 
 pickup_community_area      | 8.38  
 dropoff_community_area     | 8.92  
 fare                       | 0.17  
 tip                        | 0.17  
 additional_charges         | 0.17  
 trip_total                 | 0.17  
 shared_trip_authorized     | 0.0   
 trips_pooled               | 0.0   
 pickup_centroid_latitude   | 8.14  
 pickup_centroid_longitude  | 8.14  
 pickup_centroid_location   | 8.14  
 dropoff_centroid_latitude  | 8.7   
 dropoff_centroid_longitude | 8.7   
 dropoff_centroid_location  | 8.7   
 percent_time_chicago       | 47.81 
 percent_distance_chicago   | 47.81 
 shared_trip_match          | 47.69 
 table                      | 0.0   
 year                       | 0.0   
 

## Missing values

- About 48% of trips are missing the census tract. The city hides tracts for privacy.
- Only about 12% are missing the community area.
- Decision: we will analyze by community area (77 areas) instead of census tract. Using tracts would throw away almost half of the data.

In [4]:
rules = {
    "seconds <= 0":              F.col("trip_seconds") <= 0,
    "miles <= 0":                F.col("trip_miles") <= 0,
    "fare <= 0":                 F.col("fare") <= 0,
    "fare is null":              F.col("fare").isNull(),
    "pct_time > 1":              F.col("percent_time_chicago") > 1,
    "pct_distance > 1":          F.col("percent_distance_chicago") > 1,
    "over 4 hours":              F.col("trip_seconds") > 4 * 3600,
    "over 100 miles":            F.col("trip_miles") > 100,
    "no pickup community area":  F.col("pickup_community_area").isNull(),
    "no dropoff community area": F.col("dropoff_community_area").isNull(),
}

total = df.count()
rows = [(name, df.filter(cond).count()) for name, cond in rules.items()]
summary = spark.createDataFrame(rows, ["rule", "rows_flagged"]) \
               .withColumn("pct", F.round(F.col("rows_flagged") / total * 100, 2))
summary.show(truncate=False)

dupes = df.groupBy("trip_id").count().filter("count > 1").count()
print("duplicate trip_ids:", dupes)

+-------------------------+------------+----+
|rule                     |rows_flagged|pct |
+-------------------------+------------+----+
|seconds <= 0             |1989        |0.0 |
|miles <= 0               |201123      |0.03|
|fare <= 0                |3817627     |0.61|
|fare is null             |1060418     |0.17|
|pct_time > 1             |114578      |0.02|
|pct_distance > 1         |6525        |0.0 |
|over 4 hours             |8312        |0.0 |
|over 100 miles           |140325      |0.02|
|no pickup community area |52648176    |8.38|
|no dropoff community area|56027195    |8.92|
+-------------------------+------------+----+



26/11/01 19:26:08 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:70)
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:44)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:34)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$driverEndpoint(BlockManagerMasterEndpoint.scala:131)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.isExecutorAlive$lzycompute$1(Blo

duplicate trip_ids: 0


## How many rows each problem affects

- Very small problems: 2 trips with 0 seconds, 11 with 0 miles, 3 longer than 4 hours, 91 longer than 100 miles.
- No duplicate trip IDs.
- Bigger problems that need a closer look before deleting anything:
  - 4.6% of trips have a fare of $0 or less
  - 14.7% have percent_distance_chicago above 1
  - about 12% have no community area

In [5]:
# 1. Are $0 fares real trips? (if trip_total > 0 and miles are short, they're rounding, not errors)
df.filter(F.col("fare") <= 0) \
  .select("trip_miles", "trip_seconds", "trip_total", "tip") \
  .describe().show(vertical=True)

# 2. How far above 1 do the percentages go?
for c in ["percent_distance_chicago", "percent_time_chicago"]:
    print(c,
          "| >1.0:", df.filter(F.col(c) > 1.0).count(),
          "| >1.1:", df.filter(F.col(c) > 1.1).count(),
          "| >1.5:", df.filter(F.col(c) > 1.5).count())

# 3. Do missing community areas line up with trips partly outside Chicago?
df.withColumn("no_area", F.col("pickup_community_area").isNull()) \
  .groupBy("no_area") \
  .agg(F.round(F.avg("percent_distance_chicago"), 3).alias("avg_pct_in_chicago"),
       F.count("*").alias("trips")) \
  .show()

26/11/01 20:00:36 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:70)
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:44)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:34)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$driverEndpoint(BlockManagerMasterEndpoint.scala:131)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.isExecutorAlive$lzycompute$1(Blo

-RECORD 0--------------------------
 summary      | count              
 trip_miles   | 3817229            
 trip_seconds | 3815905            
 trip_total   | 3817615            
 tip          | 3817627            
-RECORD 1--------------------------
 summary      | mean               
 trip_miles   | 3.7017308366880854 
 trip_seconds | 805.3611261286641  
 trip_total   | 3.925099516321086  
 tip          | 0.1450801767694958 
-RECORD 2--------------------------
 summary      | stddev             
 trip_miles   | 4.652387992683689  
 trip_seconds | 649.3200589683621  
 trip_total   | 9.022346917679648  
 tip          | 1.0003901668178512 
-RECORD 3--------------------------
 summary      | min                
 trip_miles   | 0.0                
 trip_seconds | 0.0                
 trip_total   | 0.0                
 tip          | 0.0                
-RECORD 4--------------------------
 summary      | max                
 trip_miles   | 1530.6             
 trip_seconds | 72960.0     

26/11/01 20:05:43 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:70)
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:44)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:34)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$driverEndpoint(BlockManagerMasterEndpoint.scala:131)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.isExecutorAlive$lzycompute$1(Blo

+-------+------------------+---------+
|no_area|avg_pct_in_chicago|    trips|
+-------+------------------+---------+
|   true|             0.612| 52648176|
|  false|             0.957|575541504|
+-------+------------------+---------+



## Looking closer at the big problems

**$0 fares:** First we guessed these were small fares rounded down to $0,
because the city rounds fares to the nearest $2.50. That guess was wrong:
these trips average 6.7 miles and 17 minutes, which are normal trips.
So the fare is most likely missing or not reported. The trips themselves are real.

**Percent above 1:** Almost all of these are only slightly above 1.
Only 144 are above 1.1 and only 12 are above 1.5. This looks like small rounding.

**Missing community area:** These trips are only 42% inside Chicago on average,
compared to 90% for other trips. So a missing area means the trip started
or ended outside the city.

## Cleaning rules

| Problem | What we do |
|---|---|
| 0 seconds or 0 miles | Remove |
| Longer than 4 hours or 100 miles | Remove (outliers) |
| Fare $0 or missing | Keep, but don't use for price analysis |
| Percent in Chicago above 1 | Change it to 1.0 |
| No community area | Keep for city totals, but not for area analysis |

Result: we only remove about 0.05% of rows. All other trips are kept, so we don't lose real data.

Next step: run this same notebook on the full dataset.